In [1]:
import ee
# ee.Authenticate()
ee.Initialize(project="ee-joshisur231")
import geemap
Map = geemap.Map()
from helpers import config
import pandas as pd
import ast

In [16]:
config.ROI = ee.Geometry.Polygon(
        [[[82.08671845753429, 29.376317250724064],
          [82.08671845753429, 29.313208722449943],
          [82.27288522083995, 29.313208722449943],
          [82.27288522083995, 29.376317250724064]]], None, False)


frtc_lc_ic = ee.ImageCollection("projects/ee-joshisur231/assets/landcover_frtc_2000-2022_nepal")

# Waterbody 1
# Glacier 2
# Snow 3
# Forest 4
# Riverbed 5
# Built-up area 6
# Cropland 7 
# Bare soil 8
# Bare rock 9
# Grassland 10
# Other wooded land 11
water = frtc_lc_ic.map(lambda image: image.eq(1)).sum().eq(22).rename("water").unmask(0).clip(config.ROI)
forest = frtc_lc_ic.map(lambda image: image.eq(4)).sum().eq(22).rename("forest").unmask(0).clip(config.ROI)
riverbed = frtc_lc_ic.map(lambda image: image.eq(5)).sum().eq(22).rename("riverbed").unmask(0).clip(config.ROI)
built = frtc_lc_ic.map(lambda image: image.eq(6)).sum().eq(22).rename("built").unmask(0).clip(config.ROI)
soil = frtc_lc_ic.map(lambda image: image.eq(8)).sum().eq(22).rename("soil").unmask(0).clip(config.ROI)
rock = frtc_lc_ic.map(lambda image: image.eq(9)).sum().eq(22).rename("rock").unmask(0).clip(config.ROI)
grass = frtc_lc_ic.map(lambda image: image.eq(10)).sum().eq(22).rename("grass").unmask(0).clip(config.ROI)
owl = frtc_lc_ic.map(lambda image: image.eq(11)).sum().eq(22).rename("owl").unmask(0).clip(config.ROI)

stable_img = ee.Image([water, forest, riverbed, built, soil, rock, grass, owl]) # noncrop_ic = frtc_lc_ic.map(lambda image: image.neq(7).And(image.neq(3)).rename("noncrop"))
# stable_noncrop_mask = noncrop_ic.sum().eq(22).unmask(0)

In [17]:
eroded = stable_img.focalMin(radius=4, units='pixels')
subset_bands = ["forest", "built", "soil", "rock", "grass", "owl"]
patch_size = eroded.select(subset_bands).connectedPixelCount(maxSize=10, eightConnected=True)
patch_gte10 = eroded.select(subset_bands).updateMask(patch_size.gte(10))
final_mask = patch_gte10.addBands(eroded.select(["water", "riverbed"])).unmask(0).clip(config.ROI)

In [19]:
final_mask.selfMask().reduceRegion(geometry=config.ROI, scale=30, reducer=ee.Reducer.frequencyHistogram(), maxPixels=1e13)

In [ ]:
# 1. Create a class ID image (using your original lc2022 codes)
class_id_img = final_mask.select("water").multiply(1) \
    .add(final_mask.select("forest").multiply(4)) \
    .add(final_mask.select("riverbed").multiply(5)) \
    .add(final_mask.select("built").multiply(6)) \
    .add(final_mask.select("soil").multiply(8)) \
    .add(final_mask.select("rock").multiply(9)) \
    .add(final_mask.select("grass").multiply(10)) \
    .add(final_mask.select("owl").multiply(11)) \
    .selfMask().rename("class_id")
# 2. Create a PRIORITY image (Higher integer = More rare class)
# Based on your pixel counts, we rank them from 1 (Most Common) to 8 (Rarest)
priority_img = final_mask.select("forest").multiply(1) \
    .add(final_mask.select("grass").multiply(2)) \
    .add(final_mask.select("water").multiply(3)) \
    .add(final_mask.select("rock").multiply(4)) \
    .add(final_mask.select("built").multiply(5)) \
    .add(final_mask.select("riverbed").multiply(6)) \
    .add(final_mask.select("owl").multiply(7)) \
    .add(final_mask.select("soil").multiply(8)) \
    .selfMask().rename("priority_image")
# 3. Add random noise (0.0 to 0.5) to break ties. 
# This ensures pixels of the same class don't clump together.
weight_img = priority_img.add(ee.Image.random().multiply(0.5))
# 4. Find the local maximum weight within 2000m
# Earth Engine uses a sliding 2000m window. If a rare class (e.g. owl, weight=7) 
# is near a common class (e.g. forest, weight=1), the forest pixel is overpowered!
local_max = weight_img.focalMax(radius=2000, units='meters')
# 5. A pixel is kept ONLY if its weight is exactly the local maximum
is_local_max = weight_img.eq(local_max)
# 6. Apply this local maxima mask to the original class ID image
thinned_class_img = class_id_img.updateMask(is_local_max).rename('lc2022')
# 7. Now sample! Because the image is already thinned, EVERY sampled point 
# is mathematically guaranteed to be >2000m apart!
stable_noncrop_points = thinned_class_img.stratifiedSample(
    numPoints=2000,        # You can specify the exact max points per class!
    classBand='lc2022',
    region=config.ROI,
    scale=30,
    geometries=True,
    dropNulls=True,
    tileScale = 4
)

In [23]:
# geemap.ee_export_vector_to_asset(
#     collection=stable_noncrop_points,
#     description = "stable_noncrop_points_filtered",
#     assetId = "projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered_re"
# )

In [24]:
Map.addLayer(final_mask.clip(config.ROI), {}, "final_mask")
Map.addLayer(class_id_img.clip(config.ROI), {"min": 1, "max":11, "palette": ["red", "green"]}, "class_id", False)
Map.addLayer(priority_img.clip(config.ROI), {"min": 1, "max":11, "palette": ["red", "green"]}, "priority_image", False)
Map.addLayer(stable_noncrop_points, {}, "stable", False)
Map


Map(bottom=869852.0, center=[29.355845595310743, 82.17369349352917], controls=(WidgetControl(options=['positio…

In [ ]:
# geo_region = ee.Image("projects/ee-joshisur231/assets/pa_effectiveness/geoReg_nepal").rename("geoReg")
# lc_2022 = ee.ImageCollection("projects/ee-joshisur231/assets/landcover_frtc_2000-2022_nepal").filter(ee.Filter.eq("system:index", "lc2022")).first()
# strata = lc_2022.multiply(100).add(geo_region).rename("strata")

# geo_stable_noncrop = stable_noncrop_mask_final.addBands(strata)

# area_stable_noncrop_final_mask = geo_stable_noncrop.reduceRegion(
#     reducer = ee.Reducer.sum().unweighted().group(groupField = 1, groupName = "strata"),
#     geometry = config.ROI,
#     scale = 30,
#     maxPixels = 474870364
# )
# client = area_stable_noncrop_final_mask.getInfo()
# pd.DataFrame(client["groups"]).to_csv("outputs/test/geoReg_LC_nonCrop.csv", index=False)

In [ ]:
# geemap.ee_export_image_to_drive(
#     image=stable_noncrop_mask_final.selfMask(),
#     description="stable_noncrop_mask",
#     scale=30,
#     maxPixels= 992227573612
# )

In [16]:
filtered_points = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered")
roi = ee.FeatureCollection("projects/ee-joshisur231/assets/pa_effectiveness/nepal_boundary"),
filtered_points = filtered_points.merge(geometry).merge(geometry2)
frtc_ic = ee.ImageCollection("projects/ee-joshisur231/assets/landcover_frtc_2000-2022_nepal")
lc= frtc_ic.filter(ee.Filter.eq("system:index", "lc2022")).first()
  
points_lc = lc.reduceRegions(collection= filtered_points, reducer= ee.Reducer.first(), scale=30).map(lambda feat : feat.set("lc2022", ee.Number(feat.get("first")).toInt())).select(propertySelectors=["noncrop", "lc2022"], retainGeometry= True)

In [17]:
points_lc.aggregate_histogram("lc2022")

In [20]:
geemap.ee_export_vector_to_asset(
    collection= points_lc,
    description = "nonCriop",
    assetId = "projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered_withLC"
)

projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered_withLC
Exporting nonCriop... Please check the Task Manager from the JavaScript Code Editor.


## Extract NDVI

In [25]:
import ee
ee.Initialize(project="ee-joshisur231")
import geemap
import helpers.config as config

In [26]:
start_date = "2000-01-01"
end_date = "2022-12-31"

points = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered_re")

In [27]:
l8_ndvi_col = ee.ImageCollection("LANDSAT/COMPOSITES/C02/T1_L2_8DAY_NDVI")\
    .filterBounds(config.ROI)\
    .filterDate(start_date, end_date)
    
def extract_point_values(image):
    l_with_time_band = image.addBands(image.metadata("system:time_start").rename("time"))
    points_with_ndvi = l_with_time_band.reduceRegions(collection = points, scale=30, reducer=ee.Reducer.first())
    return points_with_ndvi

points_with_ndvi = l8_ndvi_col.map(extract_point_values).flatten()

In [28]:
geemap.ee_export_vector_to_drive(
    collection=points_with_ndvi,
    description="stable_nonCrop_withNDVI_re",
    fileFormat = "CSV"
)

Exporting stable_nonCrop_withNDVI_re... Please check the Task Manager from the JavaScript Code Editor.
